# 01 API Smoke TestAssumption: API is running at `http://127.0.0.1:8000`.This notebook covers create user -> alias search/select log -> summary -> optimize -> infeasible check.

In [ ]:
import jsonfrom datetime import datetime, timezonefrom urllib.request import Request, urlopenfrom urllib.error import HTTPErrorBASE_URL = "http://127.0.0.1:8000"def api(method, path, payload=None):    url = f"{BASE_URL}{path}"    body = None    headers = {"Accept": "application/json"}    if payload is not None:        body = json.dumps(payload).encode("utf-8")        headers["Content-Type"] = "application/json"    req = Request(url, data=body, headers=headers, method=method.upper())    try:        with urlopen(req, timeout=20) as resp:            raw = resp.read().decode("utf-8")            return resp.status, (json.loads(raw) if raw else {})    except HTTPError as exc:        raw = exc.read().decode("utf-8")        try:            payload = json.loads(raw)        except Exception:            payload = {"raw": raw}        return exc.code, payloaddef show(title, obj):    print(f"\\n=== {title} ===")    print(json.dumps(obj, indent=2, ensure_ascii=False))

In [ ]:
# 1) Create userstatus, user = api("POST", "/v1/users", {"name": "smoke-user"})assert status == 200, (status, user)user_id = user["user_id"]show("Create user", user)

In [ ]:
# 2) Set profile + derive targets + set configstatus, profile_resp = api("POST", "/v1/profile", {    "user_id": user_id,    "age": 30, "sex": "male", "height_cm": 180, "weight_kg": 85,    "activity_level": "moderate", "goal": "maintain"})assert status == 200, (status, profile_resp)status, derived = api("POST", "/v1/profile/derive-targets", {"user_id": user_id, "strictness": "normal"})assert status == 200, (status, derived)targets = derived["targets"]status, cfg = api("PUT", "/v1/config", {    "user_id": user_id,    "config": {        "horizon_days": 1,        "constraints": {            "calories_kcal": {"min": 700, "max": 1100},            "protein_g": {"min": 60, "max": 130},            "carbs_g": {"min": 50, "max": 200},            "fat_g": {"min": 15, "max": 70},            "fiber_g": {"min": 8},            "sat_fat_g": {"max": 20},            "sodium_mg": {"max": 3000},            "budget_try": {"max": 250}        },        "objectives_lex": [            {                "name": "min_total_deviation",                "type": "deviation",                "targets": ["calories_kcal", "protein_g", "carbs_g", "fat_g"],                "target_values": {                    "calories_kcal": targets["calories_kcal"],                    "protein_g": targets["protein_g"],                    "carbs_g": targets["carbs_g"],                    "fat_g": targets["fat_g"]                },                "tolerance": 0.01            },            {"name": "min_cost", "type": "linear", "metric": "cost_try", "sense": "min"}        ],        "food_bounds": {"min_grams_per_food": 0, "max_grams_per_food": 400}    }})assert status == 200, (status, cfg)show("Derived targets", derived)show("Set config", cfg)

In [ ]:
# 3) logs/search with alias term: simitstatus, found = api("POST", "/v1/logs/search", {"query": "simit", "limit": 5, "provider": "openfoodfacts"})assert status == 200, (status, found)assert found["candidates"], "No candidates returned"show("logs/search simit", found)# 4) Pick first if selection_required, otherwise recommendedif found.get("selection_required", False):    chosen_food_id = found["candidates"][0]["food_id"]else:    chosen_food_id = found.get("recommended_food_id") or found["candidates"][0]["food_id"]print("Chosen food_id:", chosen_food_id)

In [ ]:
# 5) Log grams with chosen food_idstatus, logged = api("POST", "/v1/logs/select", {    "user_id": user_id,    "timestamp": datetime.now(timezone.utc).isoformat(),    "food_id": chosen_food_id,    "grams": 120.0})assert status == 200, (status, logged)show("logs/select", logged)

In [ ]:
# 6) Summary todaystatus, summary = api("GET", f"/v1/summary/today?user_id={user_id}")assert status == 200, (status, summary)show("summary/today", summary)

In [ ]:
# 7) Optimize planstatus, plan = api("POST", "/v1/plan/optimize", {"user_id": user_id, "horizon_days": 1, "meal_slots": ["lunch", "dinner", "snack"]})assert status == 200, (status, plan)show("plan/optimize", plan)print("Summary:", plan.get("today_plan_summary", ""))

In [ ]:
# 8) Force infeasible config and verify 422 + suggested_relaxationsstatus, _ = api("PUT", "/v1/config", {    "user_id": user_id,    "config": {        "horizon_days": 1,        "constraints": {"protein_g": {"min": 1000.0}, "calories_kcal": {"max": 150.0}},        "objectives_lex": [            {"name": "min_total_deviation", "type": "deviation", "targets": ["protein_g", "calories_kcal"], "target_values": {"protein_g": 1000.0, "calories_kcal": 100.0}}        ],        "food_bounds": {"min_grams_per_food": 0.0, "max_grams_per_food": 50.0}    }})assert status == 200status, inf = api("POST", "/v1/plan/optimize", {"user_id": user_id, "horizon_days": 1})assert status == 422, (status, inf)detail = inf.get("detail", {})assert detail.get("code") == "infeasible_plan", detailassert detail.get("suggested_relaxations"), detailshow("Infeasible 422 response", inf)